# 01 — Data, EDA, and frozen split

**One job:** inspect the competition data and verify the one shared split

Run from top to bottom. Slow actions are off by default.

In [ ]:
%pip install -q ultralytics pycocotools segmentation-models-pytorch

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json
import sys
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_DIR = Path("/content/drive/MyDrive/Google Colab/Solar Filament Segmentation Challenge")
sys.path.insert(0, str(PROJECT_DIR))
%load_ext autoreload
%autoreload 2

from src.config import DEFAULT_PIPELINE_SETTINGS, TRAINING_SETTINGS
from src.data import copy_raw_data_to_local, load_records, make_split, verify_or_write_split

LOCAL_DATA_DIR = copy_raw_data_to_local(PROJECT_DIR / "data/raw")
TRAIN_IMAGES_DIR = LOCAL_DATA_DIR / "train/train_images"
TEST_IMAGES_DIR = LOCAL_DATA_DIR / "test/test_images"
ANNOTATIONS_PATH = LOCAL_DATA_DIR / "train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"
SPLIT_MANIFEST_PATH = PROJECT_DIR / "data/processed/split_manifest.json"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

In [ ]:
RUN_HUMAN_AGREEMENT = False  # Slow COCO-RLE comparison; the saved result is shown below

## 1. Data inventory and integrity

Facts here are computed from the raw files, not copied from the handoff.

In [ ]:
from PIL import Image

coco, records = load_records(ANNOTATIONS_PATH, TRAIN_IMAGES_DIR)
train_images = sorted(TRAIN_IMAGES_DIR.glob("*.jpeg"))
test_images = sorted(TEST_IMAGES_DIR.glob("*.jpeg"))
sizes = {Image.open(path).size for path in train_images[:20] + test_images[:20]}
assert (len(train_images), len(test_images), len(coco["images"]), len(coco["annotations"])) == (707, 180, 1154, 8199)
assert sizes == {(2048, 2048)}

def image_metadata(paths, split):
    return pd.DataFrame({"split": split, "file": [path.name for path in paths],
                         "year": [int(path.stem[:4]) for path in paths], "station": [path.stem[-2] for path in paths]})

image_frame = pd.concat([image_metadata(train_images, "train"), image_metadata(test_images, "test")])
display(image_frame.groupby(["split", "station"]).size().unstack(fill_value=0))
display(image_frame.groupby(["split", "year"]).size().unstack(fill_value=0))
print(f"Train images: {len(train_images)}, test images: {len(test_images)}, annotator records: {len(records)}, filaments: {len(coco['annotations'])}")

## 2. Annotator structure and disagreement

In [ ]:
annotator_codes = [record["annotator_image_id"].split("-", 1)[0] for record in records]
records_by_image = pd.Series([record["physical_id"] for record in records]).value_counts()
print(f"Annotator codes: {len(set(annotator_codes))}")
print("Physical images by annotator count:")
display(records_by_image.value_counts().sort_index().rename("images").to_frame())

counts = pd.DataFrame({"physical_id": [record["physical_id"] for record in records],
                       "filaments": [len(record["annotations"]) for record in records]})
disagreement = counts.groupby("physical_id")["filaments"].agg(["count", "min", "max"])
shared = disagreement.query("count > 1")
print(f"Shared images with a >=3-filament count gap: {((shared['max'] - shared['min']) >= 3).mean():.1%}")

## 3. Label-noise ceiling

The saved human-vs-human result is PQ 0.34, SQ 0.634, RQ 0.536 over 296 multi-annotator images. Recompute only when auditing the scorer.

In [ ]:
from src.metrics import human_agreement

if RUN_HUMAN_AGREEMENT:
    agreement_rows, agreement = human_agreement(records)
    display(pd.Series(agreement))
else:
    display(pd.Series({"official_pq": 0.34, "official_sq": 0.634, "official_rq": 0.536, "status": "saved verified result; recomputation skipped"}))

## 4. Frozen split verification

This is the only cell allowed to create `split_manifest.json`, and only when it is missing.

In [ ]:
manifest, train_records, validation_records = verify_or_write_split(records, SPLIT_MANIFEST_PATH, seed=42, validation_fraction=0.15)
print(f"Validation: {len(manifest['validation_physical_ids'])} physical images / {len(validation_records)} records / {sum(len(r['annotations']) for r in validation_records)} filaments")
print(f"Training: {len(manifest['train_physical_ids'])} physical images / {len(train_records)} records / {sum(len(r['annotations']) for r in train_records)} filaments")

legacy_splits = [
    PROJECT_DIR / "archive/outputs_legacy/outputs/stage2_crop_unet/split_manifest.json",
    PROJECT_DIR / "archive/outputs_legacy/outputs/stage2_crop_unet_b5/split_manifest.json",
]
for path in legacy_splits:
    if path.exists():
        assert json.loads(path.read_text()) == manifest
        print(f"Exact match: {path}")